# Games for today and tomorrow

Loads your pipeline **games** Parquet and filters rows where `game_date` is today or tomorrow (local timezone).

**Default file:** `data/games_{season}.parquet` where `season` is **`PIPELINE_SEASON`** (if set) or the **calendar year**. Run **`python -m app.main live`** (or Docker **`compose --profile live up live`**) to refresh the current season; use **`backfill --years ...`** for one-off seasons.

Legacy fallbacks: `games.parquet`, `games_live.parquet` (only if present).

**Requires** a schedule window that actually includes *today's* calendar date. If the notebook prints a `game_date` range ending in 2024 but your clock is 2026, the "today / tomorrow" filter will correctly show **0 rows** until you rebuild for the current season.

Run Jupyter with working directory `mlb-model` **or** `mlb-model/notebooks` (the next cell resolves `data/` automatically).

The load cell sets **`display.max_columns` to 50** (raise to **`None`** in that cell to show every column with no cap). The main table shows **all** columns in `sub` for today/tomorrow. A later section flags **mismatch / favorite** games using `MismatchBreakdown`-style thresholds plus optional platoon and bullpen filters.


In [8]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd

# Show many columns in DataFrame HTML / repr (raise to None for no limit).
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)

_cwd = Path.cwd()
DATA = _cwd / "data" if (_cwd / "data").is_dir() else _cwd.parent / "data"

_ps = os.environ.get("PIPELINE_SEASON", "").strip()
_season = int(_ps) if _ps.isdigit() else datetime.now().year

# Prefer games_{season}.parquet; then generic / legacy names. If several exist, use newest mtime.
_candidates = [
    DATA / f"games_{_season}.parquet",
    DATA / "games.parquet",
    DATA / "games_live.parquet",
]
_existing = [p for p in _candidates if p.is_file()]
if not _existing:
    raise FileNotFoundError(
        f"No games parquet in {DATA} (tried games_{_season}.parquet, games.parquet, games_live.parquet)"
    )
PARQUET = max(_existing, key=lambda p: p.stat().st_mtime)
if len(_existing) > 1:
    print("Multiple game parquets found; using newest by file time:", PARQUET.name)

print("Using:", PARQUET.resolve())
df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"]).dt.normalize()
gd_min, gd_max = df["game_date"].min(), df["game_date"].max()
print(f"game_date range: {gd_min.date()} .. {gd_max.date()}  ({len(df)} rows)")
if "stats_season" in df.columns:
    ss = df["stats_season"].dropna().unique()
    print("stats_season values:", sorted(ss.tolist()))
clock = pd.Timestamp.today().normalize()
if clock < gd_min or clock > gd_max:
    print(
        f"\n>>> Today's date ({clock.date()}) is outside this file's game_date range.\n"
        "    Re-run the pipeline for the season you care about, e.g. (host):\n"
        "    python -m app.main live --season 2026\n"
        "    Or: docker compose --profile live up live  (set PIPELINE_SEASON if needed)"
    )

Multiple game parquets found; using newest by file time: games_2026.parquet
Using: C:\Users\Austin\baseball\mlb-model\data\games_2026.parquet
game_date range: 2026-03-01 .. 2026-09-27  (2796 rows)
stats_season values: [2026]


In [9]:
# df already loaded and game_date normalized in previous cell
today = pd.Timestamp.today().normalize()
tomorrow = today + pd.Timedelta(days=1)
# sub = df[df["game_date"].isin([today, tomorrow])].copy()
sub = df[df["game_date"].isin([today])].copy()

print(f"Today={today.date()}, tomorrow={tomorrow.date()}, rows={len(sub)}")

Today=2026-04-14, tomorrow=2026-04-15, rows=15


In [10]:
# All columns for today/tomorrow (respects display.max_columns above).
view = sub.sort_values(["game_date", "home_team_name"])

_datapoints = [
    "game_date",
    "home_team_name",
    "away_team_name",
    # batting stats
    "home_wrc_plus",
    "away_wrc_plus",
    "offense_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "offense_platoon_diff",
    # starting pitcher stats
    "home_probable_pitcher",
    "home_sp_throws",
    "home_sp_kbb",
    "home_sp_kbb_roll14",
    "home_sp_xfip",
    "home_sp_xfip_roll14",
    "away_probable_pitcher",
    "away_sp_throws",
    "away_sp_kbb",
    "away_sp_kbb_roll14",
    "away_sp_xfip",
    "away_sp_xfip_roll14"
    "sp_kbb_diff",
    "sp_xfip_diff",
    #bullpen stats
    "home_pen_season_era",
    "home_pen_roll14_era",
    "home_pen_season_fip",
    "home_pen_roll14_fip",
    "home_pen_roll14_minus_season_fip",
    "away_pen_season_era",
    "away_pen_roll14_era",
    "away_pen_season_fip",
    "away_pen_roll14_fip",
    "away_pen_roll14_minus_season_fip",	
    # kalshi
    "kalshi_home_implied",
    "kalshi_away_implied",	
    "edge_vs_model"
]
    

data = [c for c in _datapoints if c in view.columns]
ids = [c for c in view.columns if c not in _datapoints]

organized_view = view[data + ids]
organized_view


,game_date,home_team_name,away_team_name,home_wrc_plus,away_wrc_plus,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_probable_pitcher,home_sp_throws,home_sp_kbb,home_sp_kbb_roll14,home_sp_xfip,home_sp_xfip_roll14,away_probable_pitcher,away_sp_throws,away_sp_kbb,away_sp_kbb_roll14,away_sp_xfip,sp_xfip_diff,home_pen_season_era,home_pen_roll14_era,home_pen_season_fip,home_pen_roll14_fip,home_pen_roll14_minus_season_fip,away_pen_season_era,away_pen_roll14_era,away_pen_season_fip,away_pen_roll14_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,game_pk,detailed_state,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,sp_kbb_diff,away_sp_xfip_roll14,stats_season
622,2026-04-14,Athletics,Texas Rangers,93.750000,99.752287,-6.002287,83.441916,80.886038,2.555879,Jeffrey Springs,L,13.235294,21.739130,2.445455,1.792308,MacKenzie Gore,L,31.746032,39.024390,2.732653,-0.287199,4.534351,4.645161,4.588550,4.680645,0.092096,2.006757,2.700000,3.809459,4.270000,0.460541,0.465,0.535,NaN,825022,Pre-Game,133,140,NaN,NaN,605488,669022,NaN,True,2026,2026,-18.510738,2.736364,2026
615,2026-04-14,Atlanta Braves,Miami Marlins,112.185595,105.182927,7.002668,112.406913,111.985387,0.421526,Reynaldo López,R,12.698413,17.073171,5.078723,4.962069,Max Meyer,R,10.447761,9.090909,4.463636,0.615087,0.801980,1.094595,2.238614,2.532432,0.293819,3.532710,4.743243,3.156075,3.951351,0.795277,0.605,0.395,NaN,824935,Pre-Game,144,146,0.0,0.0,625643,676974,NaN,True,2026,2026,2.250652,5.582759,2026
609,2026-04-14,Baltimore Orioles,Arizona Diamondbacks,105.040015,95.607851,9.432165,102.430799,105.542748,-3.111949,Trevor Rogers,L,11.842105,16.326531,2.573684,1.850000,Merrill Kelly,R,15.909091,NaN,3.725000,-1.151316,3.731707,4.101266,3.563415,4.505063,0.941649,5.360294,4.064516,4.335294,3.712903,-0.622391,0.595,0.405,NaN,824853,Warmup,110,109,0.0,0.0,669432,518876,NaN,True,2026,2025,-4.066986,NaN,2026
617,2026-04-14,Chicago White Sox,Tampa Bay Rays,83.460366,101.467226,-18.006860,97.724767,99.679262,-1.954495,Noah Schultz,L,NaN,NaN,NaN,NaN,Shane McClanahan,L,5.555556,5.555556,3.446154,NaN,6.590551,4.554217,5.320472,5.051807,-0.268665,6.982759,6.835443,5.841379,6.555696,0.714317,0.445,0.555,NaN,824616,Pre-Game,145,139,0.0,0.0,702273,663556,NaN,False,<NA>,2026,NaN,3.446154,2026
612,2026-04-14,Cincinnati Reds,San Francisco Giants,89.033918,92.177973,-3.144055,100.731683,85.991288,14.740394,Brady Singer,R,16.949153,17.948718,4.128571,3.100000,Robbie Ray,L,17.391304,16.666667,3.561538,0.567033,2.842105,3.030612,4.007895,4.171429,0.163534,4.395349,5.597561,4.448837,4.270732,-0.178106,0.515,0.485,NaN,824531,Warmup,113,137,0.0,0.0,663903,592662,NaN,True,2026,2026,-0.442152,3.350000,2026
610,2026-04-14,Detroit Tigers,Kansas City Royals,99.609375,93.607088,6.002287,93.064047,91.410244,1.653804,Framber Valdez,L,8.974359,5.660377,2.747059,3.190909,Cole Ragans,L,19.565217,32.000000,5.631250,-2.884191,3.701613,3.951220,4.503226,4.417073,-0.086153,5.192308,6.576923,5.292308,5.869231,0.576923,0.520,0.480,NaN,824292,Pre-Game,116,118,0.0,0.0,664285,666142,NaN,True,2026,2026,-10.590858,1.000000,2026
620,2026-04-14,Houston Astros,Colorado Rockies,115.043826,96.322409,18.721418,113.249965,85.396412,27.853553,Colton Gordon,L,13.984169,NaN,5.332558,NaN,Michael Lorenzen,R,8.000000,4.081633,5.957143,-0.624585,7.006329,8.628866,6.517722,6.965979,0.448258,4.180645,4.061947,4.570968,4.029204,-0.541764,0.615,0.385,NaN,824210,Pre-Game,117,115,0.0,0.0,676467,547179,NaN,True,2025,2026,5.984169,6.100000,2026
623,2026-04-14,Los Angeles Dodgers,New York Mets,122.903963,90.748857,32.155107,120.275397,87.677392,32.598005,Yoshinobu Yamamoto,R,17.910448,13.043478,3.488889,3.600000,Nolan McLean,R,21.875000,10.526316,2.740000,0.748889,2.675676,4.068493,3.289189,4.045205,0.756016,1.711268,1.534091,3.247887,3.236364,-0.011524,

## Pitch / offense mismatches ("favorites")

Base criteria from `MismatchBreakdown.ipynb` (SP K-BB edge, SP xFIP edge, offense edge), but this notebook now supports **N-of-3 matching**:

- Example default: **at least 2 of 3** core criteria for home-favored or away-favored.
- Optional extras can still be layered on:
  - **Platoon:** `offense_platoon_diff` agrees with favored side.
  - **Bullpen:** favored side has lower `*_pen_roll14_fip`.

The output also includes a simple **`mismatch_score`** so you can rank candidates while we refine a fuller scoring model.

In [18]:
# --- Core thresholds from MismatchBreakdown.ipynb ---
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2  # out of 3 core checks (2 = "most", 3 = strict old behavior)

# --- Optional: platoon + bullpen roll14 (lower FIP = better pen) ---
USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0  # require |offense_platoon_diff| > this when extras on
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0  # favored pen FIP must be lower by this much vs opponent pen

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks for each side
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    # Home favored: home pen better (lower FIP) than away
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    # Away favored: away pen better than home
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

# Keep ALL rows so you can inspect near-misses, and add match flags.
# (Old strict filtering is intentionally disabled.)
favorites = s.copy().sort_values(["game_date", "home_team_name"])

favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

# Preferred side for scoring / display = stronger core count (tie -> absolute edge blend).
_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = favorites.apply(
    lambda r: r["home_team_name"] if r["_mismatch_side"] == "home_favored" else r["away_team_name"],
    axis=1,
)

# Basic mismatch score (higher = stronger favorite setup)
favorites["mismatch_score"] = (
    (favorites["sp_kbb_diff"].abs() / SP_KBB_ABS)
    + (favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS)
    + (favorites["offense_diff"].abs() / OFFENSE_ABS)
)
if "offense_platoon_diff" in favorites.columns:
    favorites["mismatch_score"] += 0.35 * (favorites["offense_platoon_diff"].abs() / 10.0)
if {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(favorites.columns):
    pen_gap = (favorites["away_pen_roll14_fip"] - favorites["home_pen_roll14_fip"]).where(
        favorites["_mismatch_side"] == "home_favored",
        favorites["home_pen_roll14_fip"] - favorites["away_pen_roll14_fip"],
    )
    favorites["mismatch_score"] += 0.35 * (pen_gap.fillna(0) / 0.75)

favorites = favorites.sort_values(["mismatch_score", "core_matches", "game_date"], ascending=[False, False, True])

_prio = [
    "game_date",
    "detailed_state",
    "favored_team",
    "_mismatch_side",
    "core_matches",
    "mismatch_score",
    "home_team_name",
    "away_team_name",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip",
    "home_pen_season_fip",
    "away_pen_season_fip",
    "home_probable_pitcher",
    "away_probable_pitcher",
]
_prio = [c for c in _prio if c in favorites.columns]
_rest = [c for c in favorites.columns if c not in _prio]
favorites_view = favorites[_prio + _rest]

print(
    f"Core filter (>= {MIN_CORE_MATCHES}/3): home={int(home_base.sum())}, away={int(away_base.sum())} | "
    f"Rows shown: {len(favorites)} (all today/tomorrow rows, sorted by mismatch_score)"
)

# do this if you need to drop rows, for example, rookie pitcher
favorites_view = favorites_view.drop(index=617)
favorites_view

Core filter (>= 2/3): home=3, away=6 | Rows shown: 15 (all today/tomorrow rows, sorted by mismatch_score)


,game_date,detailed_state,favored_team,_mismatch_side,core_matches,mismatch_score,home_team_name,away_team_name,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_offense_platoon,away_offense_platoon,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_season_fip,away_pen_season_fip,home_probable_pitcher,away_probable_pitcher,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras
611,2026-04-14,Delayed Start,Pittsburgh Pirates,home_favored,2,14.934482,Pittsburgh Pirates,Washington Nationals,11.259755,-4.277778,-5.716463,-5.428250,97.424075,102.852325,3.815909,7.810280,4.258621,6.869231,Mitch Keller,PJ Poulin,823398,134,120,0.0,0.0,656605,676571,R,L,104.468369,110.184832,8.695652,-2.564103,2.711111,6.988889,NaN,True,2026,2026,6.250000,0.000000,3.016667,9.300000,3.910345,6.230769,3.988636,7.570093,-0.442712,0.941050,0.625,0.375,NaN,2026,2,0,True,False,False,False
610,2026-04-14,Pre-Game,Detroit Tigers,home_favored,1,10.634454,Detroit Tigers,Kansas City Royals,-10.590858,-2.884191,6.002287,1.653804,93.064047,91.410244,4.417073,5.869231,4.503226,5.292308,Framber Valdez,Cole Ragans,824292,116,118,0.0,0.0,664285,666142,L,L,99.609375,93.607088,8.974359,19.565217,2.747059,5.631250,NaN,True,2026,2026,5.660377,32.000000,3.190909,1.000000,3.701613,5.192308,3.951220,6.576923,-0.086153,0.576923,0.520,0.480,NaN,2026,1,1,False,False,False,False
618,2026-04-14,Pre-Game,Toronto Blue Jays,away_favored,2,8.321686,Milwaukee Brewers,Toronto Blue Jays,-9.141791,2.087520,4.716082,14.612899,111.423353,96.810454,3.911765,3.662500,3.270213,3.671429,Jacob Misiorowski,Kevin Gausman,823802,158,141,0.0,0.0,694819,592332,R,R,103.753811,99.037729,28.358209,37.500000,3.283673,1.196154,NaN,True,2026,2026,23.404255,28.888889,3.364706,0.982353,2.106383,4.959184,2.541176,3.937500,0.641552,-0.008929,0.535,0.465,NaN,2026,0,2,False,True,False,False
622,2026-04-14,Pre-Game,Texas Rangers,away_favored,1,7.625962,Athletics,Texas Rangers,-18.510738,-0.287199,-6.002287,2.555879,83.441916,80.886038,4.680645,4.270000,4.588550,3.809459,Jeffrey Springs,MacKenzie Gore,825022,133,140,NaN,NaN,605488,669022,L,L,93.750000,99.752287,13.235294,31.746032,2.445455,2.732653,NaN,True,2026,2026,21.739130,39.024390,1.792308,2.736364,4.534351,2.006757,4.645161,2.700000,0.092096,0.460541,0.465,0.535,NaN,2026,0,1,False,False,False,False
623,2026-04-14,Pre-Game,New York Mets,away_favored,2,7.553196,Los Angeles Dodgers,New York Mets,-3.964552,0.748889,32.155107,32.598005,120.275397,87.677392,4.045205,3.236364,3.289189,3.247887,Yoshinobu Yamamoto,Nolan McLean,823965,119,121,0.0,0.0,808967,690997,R,R,122.903963,90.748857,17.910448,21.875000,3.488889,2.740000,NaN,True,2026,2026,13.043478,10.526316,3.600000,2.725000,2.675676,1.711268,4.068493,1.534091,0.756016,-0.011524,0.655,0.345,NaN,2026,1,2,False,True,False,False
621,2026-04-14,Pre-Game,Seattle Mariners,away_favored,2,6.767848,San Diego Padres,Seattle Mariners,-8.823529,1.655556,5.144817,0.281017,101.025713,100.744696,2.342105,2.360870,2.576510,2.822689,Michael King,Bryan Woo,823315,135,136,0.0,0.0,650633,693433,R,R,99.180640,94.035823,10.294118,19.117647,3.700000,2.044444,NaN,True,2026,2026,10.416667,11.111111,3.700000,2.766667,3.442953,2.042017,3.694737,1.173913,-0.234405,-0.461820,0.485,0.515,NaN,2026,0,2,False,True,False,False
619,2026-04-14,Pre-Game,Cleveland Guardians,away_favored,2,6.640542,St. Louis Cardinals,C

In [35]:
import numpy as np
import pandas as pd

s = favorites_view.copy()

# =========================================================
# V6 MODEL — SLATE-NORMALIZED EDGE SYSTEM
# =========================================================

# =========================================================
# 1. NORMALIZED SIGNALS (STABLE SCALING)
# =========================================================

SP_KBB_ABS = 2.5
SP_XFIP_ABS = 0.75
OFFENSE_ABS = 5.0
PLATOON_ABS = 12.0
PEN_ABS = 1.5

kbb_norm = s["sp_kbb_diff"] / SP_KBB_ABS
xfip_norm = -s["sp_xfip_diff"] / SP_XFIP_ABS
off_norm = s["offense_diff"] / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

pen_gap = s["away_pen_roll14_fip"] - s["home_pen_roll14_fip"]
pen_norm = pen_gap / PEN_ABS

# =========================================================
# 2. STRUCTURED SIGNAL COMPONENTS
# =========================================================

sp_vector = (0.80 * kbb_norm) + (0.20 * xfip_norm)
off_vector = off_norm + (0.15 * platoon_norm)
pen_vector = 0.5 * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

sp_vector = np.clip(sp_vector, -2, 2)
off_vector = np.clip(off_vector, -2, 2)
pen_vector = np.clip(pen_vector, -2, 2)

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

# =========================================================
# 3. EDGE CONSTRUCTION
# =========================================================

sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

raw_edge = mean_sign * mean_mag

# =========================================================
# 4. CONFIDENCE STRUCTURE
# =========================================================

agreement = 1 - np.mean(np.var(sign_matrix, axis=0))
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (
    0.45 * agreement +
    0.30 * direction_purity +
    0.25 * mag_consistency
)

instability = np.std(signal_matrix, axis=0)
instability_penalty = 1 / (1 + instability)

risk_penalty = np.tanh(
    0.35 * (np.sign(sp_vector) != np.sign(off_vector)).astype(int) +
    0.45 * instability +
    0.20 * np.abs(sp_vector - off_vector)
)

trap_score = np.abs(raw_edge) * (1 - coherence_score)

quality_score = (
    raw_edge *
    coherence_score *
    instability_penalty *
    (1 / (1 + risk_penalty))
)

risk_adjusted_edge = quality_score * np.exp(-trap_score)

# =========================================================
# 5. SLATE NORMALIZATION
# =========================================================

s["edge_rank_pct"] = risk_adjusted_edge.rank(pct=True)
s["edge_z"] = (risk_adjusted_edge - risk_adjusted_edge.mean()) / (risk_adjusted_edge.std() + 1e-9)

# =========================================================
# 6. DECISION LAYER (SLATE RELATIVE)
# =========================================================

s["decision"] = np.select(
    [
        s["edge_rank_pct"] >= 0.90,
        s["edge_rank_pct"] >= 0.75
    ],
    ["BET", "LEAN"],
    default="NO BET"
)

# =========================================================
# 7. TIERS
# =========================================================

s["tier"] = np.select(
    [
        (s["decision"] == "BET") & (s["edge_rank_pct"] >= 0.97),
        (s["decision"] == "BET")
    ],
    ["A (Strong Bet)", "B (Playable Bet)"],
    default=np.where(
        s["decision"] == "LEAN",
        "C (Lean)",
        "D (Pass)"
    )
)

# =========================================================
# 8. TEAM LOGIC
# =========================================================

prefer_home = sp_vector >= 0

s["favored_team"] = np.where(
    prefer_home,
    s["home_team_name"],
    s["away_team_name"]
)

s["_mismatch_side"] = np.where(
    prefer_home,
    "home_favored",
    "away_favored"
)

# =========================================================
# 9. INTERPRETABILITY
# =========================================================

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = (np.sum(sign_matrix < 0, axis=0) > 0).astype(int)
s["instability"] = instability

s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score

# =========================================================
# 10. FEATURES
# =========================================================

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int) +
    (np.abs(xfip_norm) > 1).astype(int) +
    (np.abs(off_norm) > 1).astype(int)
)

# =========================================================
# 11. MARKET
# =========================================================

if "kalshi_home_implied" in s.columns:
    s["implied_prob"] = np.where(
        prefer_home,
        s["kalshi_home_implied"],
        s["kalshi_away_implied"]
    )
    s["market_edge"] = risk_adjusted_edge - s["implied_prob"]

# =========================================================
# 12. SORT
# =========================================================

s = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False])

# =========================================================
# 13. OUTPUT
# =========================================================

priority_cols = [
    "game_date",
    "home_team_name",
    "away_team_name",
    "favored_team",
    "_mismatch_side",
    "risk_adjusted_edge",
    "edge_rank_pct",
    "edge_z",
    "decision",
    "tier",
    "coherence_score",
    "trap_score",
    "instability",
    "signal_agreement",
    "signal_conflict",
    "direction_conflict",
    "core_matches",
    "home_probable_pitcher",
    "away_probable_pitcher",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip"
]

priority_cols = [c for c in priority_cols if c in s.columns]
rest_cols = [c for c in s.columns if c not in priority_cols]

final_view = s[priority_cols + rest_cols]

print(f"V6 complete — games evaluated: {len(final_view)}")
final_view


V6 complete — games evaluated: 14


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
615,2026-04-14,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.189448,1.000000,2.424275,BET,A (Strong Bet),0.611585,0.315258,0.421498,3,0,0,1,Reynaldo López,Max Meyer,2.250652,0.615087,7.002668,0.421526,2.532432,3.951351,Pre-Game,2.033249,112.406913,111.985387,2.238614,3.156075,824935,144,146,0.0,0.0,625643,676974,R,R,112.185595,105.182927,12.698413,10.447761,5.078723,4.463636,NaN,True,2026,2026,17.073171,9.090909,4.962069,5.582759,0.801980,3.532710,1.094595,4.743243,0.293819,0.795277,0.605,0.395,NaN,2026,0,1,False,False,False,False,0.259660,0.605,-0.415552
620,2026-04-14,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.034786,0.928571,0.481358,BET,B (Playable Bet),0.339696,0.365290,1.404279,2,1,1,2,Colton Gordon,Michael Lorenzen,5.984169,-0.624585,18.721418,27.853553,6.965979,4.029204,Pre-Game,4.720413,113.249965,85.396412,6.517722,4.570968,824210,117,115,0.0,0.0,676467,547179,L,R,115.043826,96.322409,13.984169,8.000000,5.332558,5.957143,NaN,True,2025,2026,NaN,4.081633,NaN,6.100000,7.006329,4.180645,8.628866,4.061947,0.448258,-0.541764,0.615,0.385,NaN,2026,3,0,True,False,False,False,0.050124,0.615,-0.580214
614,2026-04-14,New York Yankees,Los Angeles Angels,New York Yankees,home_favored,0.028219,0.857143,0.398858,LEAN,C (Lean),0.382722,0.160372,0.700588,2,1,1,2,Ryan Weathers,Reid Detmers,3.175618,-1.503989,0.571646,-32.925729,2.020000,3.891209,Pre-Game,6.149314,76.225318,109.151047,2.000000,3.803448,823562,147,108,0.0,0.0,677960,672282,L,L,99.895198,99.323552,19.117647,15.942029,1.787500,3.291489,NaN,True,2026,2026,16.326531,4.166667,1.985714,4.736364,2.700000,3.351724,3.960000,3.560440,0.020000,0.087761,0.615,0.385,NaN,2026,2,0,True,False,False,False,0.033127,0.615,-0.586781
611,2026-04-14,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.026879,0.785714,0.382028,LEAN,C (Lean),0.340608,0.332817,1.383363,2,1,1,3,Mitch Keller,PJ Poulin,11.259755,-4.277778,-5.716463,-5.428250,3.815909,7.810280,Delayed Start,14.934482,97.424075,102.852325,4.258621,6.869231,823398,134,120,0.0,0.0,656605,676571,R,L,104.468369,110.184832,8.695652,-2.564103,2.711111,6.988889,NaN,True,2026,2026,6.250000,0.000000,3.016667,9.300000,3.910345,6.230769,3.988636,7.570093,-0.442712,0.941050,0.625,0.375,NaN,2026,2,0,True,False,False,False,0.037493,0.625,-0.598121
610,2026-04-14,Detroit Tigers,Kansas City Royals,Kansas City Royals,away_favored,0.023383,0.714286,0.338113,NO BET,D (Pass),0.340846,0.271365,1.377981,2,1,1,3,Framber Valdez,Cole Ragans,-10.590858,-2.884191,6.002287,1.653804,4.417073,5.869231,Pre-Game,10.634454,93.064047,91.410244,4.503226,5.292308,824292,116,118,0.0,0.0,664285,666142,L,L,99.609375,93.607088,8.974359,19.565217,2.747059,5.631250,NaN,True,2026,2026,5.660377,32.000000,3.190909,1.000000,3.701613,5.192308,3.95

In [33]:
def decision_layer(df):

    conditions = [
        # BET (strong + clean)
        (
            (df["risk_adjusted_edge"] > 1.0) &
            (df["coherence_score"] > 0.60) &
            (df["instability"] < 1.2)
        ),

        # LEAN (some edge but imperfect structure)
        (
            (df["risk_adjusted_edge"] > 0.5) &
            (df["coherence_score"] > 0.45)
        ),

        # NO BET (everything else)
    ]

    choices = ["BET", "LEAN"]

    df["decision"] = np.select(conditions, choices, default="NO BET")

    return df

decision_layer(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,quality_score,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,implied_prob,market_edge,decision
615,2026-04-14,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.757791,0.259660,0.611585,0.315258,0.421498,3,0,0,1,Reynaldo López,Max Meyer,2.250652,0.615087,7.002668,0.421526,2.532432,3.951351,Pre-Game,2.033249,112.406913,111.985387,2.238614,3.156075,824935,144,146,0.0,0.0,625643,676974,R,R,112.185595,105.182927,12.698413,10.447761,5.078723,4.463636,NaN,True,2026,2026,17.073171,9.090909,4.962069,5.582759,0.801980,3.532710,1.094595,4.743243,0.293819,0.795277,0.605,0.395,NaN,2026,0,1,False,False,False,False,0.605,0.152791,LEAN
620,2026-04-14,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.116048,0.041804,0.339696,0.365290,1.404279,2,1,2,2,Colton Gordon,Michael Lorenzen,5.984169,-0.624585,18.721418,27.853553,6.965979,4.029204,Pre-Game,4.720413,113.249965,85.396412,6.517722,4.570968,824210,117,115,0.0,0.0,676467,547179,L,R,115.043826,96.322409,13.984169,8.000000,5.332558,5.957143,NaN,True,2025,2026,NaN,4.081633,NaN,6.100000,7.006329,4.180645,8.628866,4.061947,0.448258,-0.541764,0.615,0.385,NaN,2026,3,0,True,False,False,False,0.615,-0.498952,NO BET
614,2026-04-14,New York Yankees,Los Angeles Angels,New York Yankees,home_favored,0.106198,0.031168,0.382722,0.160372,0.700588,2,1,2,2,Ryan Weathers,Reid Detmers,3.175618,-1.503989,0.571646,-32.925729,2.020000,3.891209,Pre-Game,6.149314,76.225318,109.151047,2.000000,3.803448,823562,147,108,0.0,0.0,677960,672282,L,L,99.895198,99.323552,19.117647,15.942029,1.787500,3.291489,NaN,True,2026,2026,16.326531,4.166667,1.985714,4.736364,2.700000,3.351724,3.960000,3.560440,0.020000,0.087761,0.615,0.385,NaN,2026,2,0,True,False,False,False,0.615,-0.508802,NO BET
611,2026-04-14,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.105455,0.036775,0.340608,0.332817,1.383363,2,1,2,3,Mitch Keller,PJ Poulin,11.259755,-4.277778,-5.716463,-5.428250,3.815909,7.810280,Delayed Start,14.934482,97.424075,102.852325,4.258621,6.869231,823398,134,120,0.0,0.0,656605,676571,R,L,104.468369,110.184832,8.695652,-2.564103,2.711111,6.988889,NaN,True,2026,2026,6.250000,0.000000,3.016667,9.300000,3.910345,6.230769,3.988636,7.570093,-0.442712,0.941050,0.625,0.375,NaN,2026,2,0,True,False,False,False,0.625,-0.519545,NO BET
610,2026-04-14,Detroit Tigers,Kansas City Royals,Kansas City Royals,away_favored,0.091739,0.030085,0.340846,0.271365,1.377981,2,1,2,3,Framber Valdez,Cole Ragans,-10.590858,-2.884191,6.002287,1.653804,4.417073,5.869231,Pre-Game,10.634454,93.064047,91.410244,4.503226,5.292308,824292,116,118,0.0,0.0,664285,666142,L,L,99.609375,93.607088,8.974359,19.565217,2.747059,5.631250,NaN,True,2026,2026,5.660377,32.000000,3.190909,1.000000,3.701613,5.192308,3.951220,6.576923,-0.086153,0.576923,0.520,0.480,NaN,2026,1,1,False,False,False,False,0.480,-0.388261,NO BET
613,2026-04-14,Philadelphia Phillies,Chicago Cubs,Chicago Cub

In [34]:
def confidence_tier(df):

    df["tier"] = np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.75),
        "A (Strong Bet)",
        np.where(
            (df["decision"] == "BET"),
            "B (Playable Bet)",
            np.where(
                (df["decision"] == "LEAN"),
                "C (Lean Only)",
                "D (Pass)"
            )
        )
    )

    return df

confidence_tier(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,quality_score,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,implied_prob,market_edge,decision,tier
615,2026-04-14,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.757791,0.259660,0.611585,0.315258,0.421498,3,0,0,1,Reynaldo López,Max Meyer,2.250652,0.615087,7.002668,0.421526,2.532432,3.951351,Pre-Game,2.033249,112.406913,111.985387,2.238614,3.156075,824935,144,146,0.0,0.0,625643,676974,R,R,112.185595,105.182927,12.698413,10.447761,5.078723,4.463636,NaN,True,2026,2026,17.073171,9.090909,4.962069,5.582759,0.801980,3.532710,1.094595,4.743243,0.293819,0.795277,0.605,0.395,NaN,2026,0,1,False,False,False,False,0.605,0.152791,LEAN,C (Lean Only)
620,2026-04-14,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.116048,0.041804,0.339696,0.365290,1.404279,2,1,2,2,Colton Gordon,Michael Lorenzen,5.984169,-0.624585,18.721418,27.853553,6.965979,4.029204,Pre-Game,4.720413,113.249965,85.396412,6.517722,4.570968,824210,117,115,0.0,0.0,676467,547179,L,R,115.043826,96.322409,13.984169,8.000000,5.332558,5.957143,NaN,True,2025,2026,NaN,4.081633,NaN,6.100000,7.006329,4.180645,8.628866,4.061947,0.448258,-0.541764,0.615,0.385,NaN,2026,3,0,True,False,False,False,0.615,-0.498952,NO BET,D (Pass)
614,2026-04-14,New York Yankees,Los Angeles Angels,New York Yankees,home_favored,0.106198,0.031168,0.382722,0.160372,0.700588,2,1,2,2,Ryan Weathers,Reid Detmers,3.175618,-1.503989,0.571646,-32.925729,2.020000,3.891209,Pre-Game,6.149314,76.225318,109.151047,2.000000,3.803448,823562,147,108,0.0,0.0,677960,672282,L,L,99.895198,99.323552,19.117647,15.942029,1.787500,3.291489,NaN,True,2026,2026,16.326531,4.166667,1.985714,4.736364,2.700000,3.351724,3.960000,3.560440,0.020000,0.087761,0.615,0.385,NaN,2026,2,0,True,False,False,False,0.615,-0.508802,NO BET,D (Pass)
611,2026-04-14,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,0.105455,0.036775,0.340608,0.332817,1.383363,2,1,2,3,Mitch Keller,PJ Poulin,11.259755,-4.277778,-5.716463,-5.428250,3.815909,7.810280,Delayed Start,14.934482,97.424075,102.852325,4.258621,6.869231,823398,134,120,0.0,0.0,656605,676571,R,L,104.468369,110.184832,8.695652,-2.564103,2.711111,6.988889,NaN,True,2026,2026,6.250000,0.000000,3.016667,9.300000,3.910345,6.230769,3.988636,7.570093,-0.442712,0.941050,0.625,0.375,NaN,2026,2,0,True,False,False,False,0.625,-0.519545,NO BET,D (Pass)
610,2026-04-14,Detroit Tigers,Kansas City Royals,Kansas City Royals,away_favored,0.091739,0.030085,0.340846,0.271365,1.377981,2,1,2,3,Framber Valdez,Cole Ragans,-10.590858,-2.884191,6.002287,1.653804,4.417073,5.869231,Pre-Game,10.634454,93.064047,91.410244,4.503226,5.292308,824292,116,118,0.0,0.0,664285,666142,L,L,99.609375,93.607088,8.974359,19.565217,2.747059,5.631250,NaN,True,2026,2026,5.660377,32.000000,3.190909,1.000000,3.701613,5.192308,3.951220,6.576923,-0.086153,0.576923,0.520,0.480,NaN,2026,1,1,False,False,False,False,0.480,-0.388261,NO BET,D (Pass)
613,20